In [2]:
import sys
sys.path.append("../")  # go up one level from notebooks/ to project root (for using src/)
import os
from dotenv import load_dotenv
import json
import requests
from src.query_builder import build_semantic_scholar_query, load_keywords
from src.data_collection import collect_semantic_scholar_data, load_config

In [3]:
# Load API keys from .env file
load_dotenv("../.env")
semantic_scholar_api_key = os.getenv("SEMANTIC_SCHOLAR_API_KEY")

# Semantic Scholar

# Didactic example
Didactic example of how to use the bulk search endpoint. 

In [4]:
# Semantic Scholar didactic example of how to use the bulk search endpoint

# Define the bulk search endpoint
url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"

# Set up your query parameters
query_params = {
    "query": '"climate change" AND "social media" AND (misinformation OR polarization)',
    # "year": "2020",
    "publicationDateOrYear": "2020-01-01:2020-12-31", 
    "fields": "paperId,title,year"
    # "fields": "paperId,externalIds,title,year,publicationDate,abstract,publicationVenue,journal,citationCount,s2FieldsOfStudy,publicationTypes,authors"
}

# Optional: Add your API key if you have one to avoid rate limits
headers = {"x-api-key": semantic_scholar_api_key}

try:
    response = requests.get(url, params=query_params, headers=headers)
    response.raise_for_status()

    # Parse and print the JSON response
    data = response.json()
    print(json.dumps(data, indent=2))

except requests.exceptions.HTTPError as err:
    print(f"HTTP Error: {err}")
except Exception as e:
    print(f"An error occurred: {e}")

{
  "total": 2,
  "token": null,
  "data": [
    {
      "paperId": "0978a91dbbfdf0b9762fc572e9f7ff3b73b25add",
      "title": "Partisanship, Impeachment, and the Democratic Primaries: American Political Discourse, January - February 2020",
      "year": 2020
    },
    {
      "paperId": "85f4d04076aefafe60a8ead0435b34bf7d8fbfdd",
      "title": "Online misinformation about climate change",
      "year": 2020
    }
  ]
}


## Extract Semantic Scholar data

These are the main papers that we want to analyse. 
They belong to the following subset:
```climate AND nlp AND social_media AND (misinformation OR polarization)```

In [4]:
config = load_config("../config.yml")

ss_config = config["semantic_scholar"]
date_start = config["search"]["date_start"]
date_end = config["search"]["date_end"]

In [5]:
# Build the query

logic = "climate AND nlp AND social_media AND (misinformation OR polarization)"

semantic_scholar_query = build_semantic_scholar_query(
    logic_string=logic,
    keywords_path="../data/search_query/keywords.yml"
)

print(semantic_scholar_query)
print("\nNumber of characters", len(semantic_scholar_query))
print("Semantic Scholar query length limit is 2000 characters") 

("global warming" | "climate change" | "climate science" | "climate denial" | "climate crisis" | "climate skeptic" | "climate skepticism" | "climate skeptics" | "climate skepticist" | "climate action" | "climate misinformation" | "climate disinformation") + ("natural language processing" | NLP | "machine learning" | LLM | AI | "artificial intelligence" | "language model" | "language models" | "neural network" | "neural networks" | GenAI | "Generative AI" | "text mining" | "text processing" | "text classification" | "deep learning" | "transformer*" | "argument mining" | "sentiment analysis") + ("social media" | "social network" | "social networks" | "social platform" | "social platforms" | "online community" | "online communities" | "alternative platform" | "alternative platforms" | alt-tech | Youtube | Rumble | Bitchute | Odysee | Twitch | Quora | Facebook | Instagram | Tiktok | Whatsapp | Telegram | Signal | WeChat | Discord | Threads | Mastodon | Reddit | X | Twitter/x | Twitter | Bl

In [ ]:
collect_semantic_scholar_data(
    query=semantic_scholar_query,
    fields=ss_config["fields_full"],
    output_path=f"../data/raw/semantic_scholar/semantic_scholar_data.jsonl",
    date_start=date_start,
    date_end=date_end,
    api_key=semantic_scholar_api_key
)

{'query': '("global warming" | "climate change" | "climate science" | "climate denial" | "climate crisis" | "climate skeptic" | "climate skepticism" | "climate skeptics" | "climate skepticist" | "climate action" | "climate misinformation" | "climate disinformation") + ("natural language processing" | NLP | "machine learning" | LLM | AI | "artificial intelligence" | "language model" | "language models" | "neural network" | "neural networks" | GenAI | "Generative AI" | "text mining" | "text processing" | "text classification" | "deep learning" | "transformer*" | "argument mining" | "sentiment analysis") + ("social media" | "social network" | "social networks" | "social platform" | "social platforms" | "online community" | "online communities" | "alternative platform" | "alternative platforms" | alt-tech | Youtube | Rumble | Bitchute | Odysee | Twitch | Quora | Facebook | Instagram | Tiktok | Whatsapp | Telegram | Signal | WeChat | Discord | Threads | Mastodon | Reddit | X | Twitter/x | T

88

## Examine the interseccions

In [6]:
from itertools import combinations, chain
import time

sets = ["climate", "nlp", "social_media", "misinformation", "polarization"]
all_combinations = list(chain.from_iterable(combinations(sets, r) for r in range(1, len(sets) + 1)))

# Dictionary to store the results
intersections_totals = {}

for combination in all_combinations:

    # Logic
    logic = " AND ".join(combination)
    print(f"Retrieving data for: {logic}")

    #Name for the file
    file_name = "_".join(logic.split(" "))
    
    # Build the query
    semantic_scholar_query = build_semantic_scholar_query(
        logic_string=logic,
        keywords_path="../data/search_query/keywords.yml"
    )

    # Extract data
    total = collect_semantic_scholar_data(
        query=semantic_scholar_query,
        fields=ss_config["fields_lite"],
        output_path=f"../data/raw/semantic_scholar/sets/{file_name}.jsonl", 
        date_start=date_start,
        date_end=date_end,
        api_key=semantic_scholar_api_key
    )

    # Store the total number of results for this combination
    intersections_totals[logic] = total

    sleep_time = 1.1 if semantic_scholar_api_key else 3.5
    time.sleep(sleep_time)  # Sleep to respect rate limits

Retrieving data for: climate
{'query': '("global warming" | "climate change" | "climate science" | "climate denial" | "climate crisis" | "climate skeptic" | "climate skepticism" | "climate skeptics" | "climate skepticist" | "climate action" | "climate misinformation" | "climate disinformation")', 'publicationDateOrYear': '2020-01-01:2026-05-01', 'fields': 'paperId,title,year,venue,citationCount'}
Query: ("global warming" | "climate change" | "climate science" | "climate denial" | "climate crisis" | "climate skeptic" | "climate skepticism" | "climate skeptics" | "climate skepticist" | "climate action" | "climate misinformation" | "climate disinformation")
Output: ../data/raw/semantic_scholar/sets/climate.jsonl
API key: yes | Sleep: 1.1s
  Fetching batch | total so far: 0
  Fetching batch | total so far: 1000
  Fetching batch | total so far: 2000
  Fetching batch | total so far: 3000
  Fetching batch | total so far: 4000
  Fetching batch | total so far: 5000
  Fetching batch | total so f

In [ ]:
# Save the totals to a JSON file
with open("../data/raw/semantic_scholar/sets/intersections_totals.json", "w") as f:
    json.dump(intersections_totals, f, indent=2)

# Load the totals from the JSON file
# with open("../data/raw/semantic_scholar/sets/intersections_totals.json", "r") as f:
    # loaded_totals = json.load(f)

In [10]:
from matplotlib_venn import venn3
venn3(subsets = (1, 1, 1, 2, 1, 2, 2), set_labels = ('Set1', 'Set2', 'Set3'))

ModuleNotFoundError: No module named 'matplotlib_venn'